# 0b. Pushforward Maps and Distances Between Distributions

Notebook 0a built the *mechanism* (kernels, mean shift, particle flows). This notebook builds the *goal*.

Generative modelling has a one-line statement:

> find a map $f$ such that pushing a simple prior through it produces the data distribution.

Everything — GANs, VAEs, normalising flows, diffusion, flow matching, drifting — is a different answer to
*how do you find that $f$*. Getting crisp about the pushforward, and about what it means to measure a distance
between two distributions you can only *sample*, is what makes the drifting loss look inevitable rather than arbitrary.

In [ ]:
import math
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110

## 1. The pushforward

Let $\boldsymbol{\epsilon}\sim p_{\boldsymbol{\epsilon}}$ be a simple random variable (say $\mathcal{N}(0,I)$ in $\mathbb{R}^C$),
and let $f:\mathbb{R}^C\to\mathbb{R}^D$ be any map. Then $\mathbf{x}=f(\boldsymbol{\epsilon})$ is a random variable with
some distribution $q$, written

$$q = f_{\#}\,p_{\boldsymbol{\epsilon}}$$

and read "*$q$ is the pushforward of $p_{\boldsymbol{\epsilon}}$ under $f$*". Sampling from $q$ costs **one function call**:
draw $\boldsymbol{\epsilon}$, apply $f$. The density of $q$, on the other hand, is usually intractable — which is exactly
why we will never need it.

Note $C$ and $D$ need not match. A drifting generator maps $32\times32\times4$ Gaussian noise to a
$32\times32\times4$ latent, but nothing stops the input from being 64-dimensional noise plus a handful of discrete
"style" indices, which is what the paper actually does.

In [ ]:
# a random 2-layer MLP is already a rich pushforward map
def random_map(seed, h=64):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(2, h), nn.Tanh(), nn.Linear(h, h), nn.Tanh(), nn.Linear(h, 2))

eps = torch.randn(4000, 2)
fig, ax = plt.subplots(1, 4, figsize=(14, 3.4))
ax[0].scatter(eps[:, 0], eps[:, 1], s=2, alpha=0.3, c="k")
ax[0].set_title("prior:  eps ~ N(0, I)")
for i, s in enumerate([1, 2, 3]):
    with torch.no_grad():
        x = random_map(s)(eps) * 2.5
    ax[i + 1].scatter(x[:, 0], x[:, 1], s=2, alpha=0.3, c="tab:orange")
    ax[i + 1].set_title(f"pushforward  q = f#(p_eps)   (seed {s})")
for a in ax: a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

Three random networks, three wildly different distributions. Training a generative model means **searching this
space of pushforwards** for one that lands on the data.

Two facts to internalise:

* A pushforward is a *deterministic* map applied to randomness. All the entropy comes from $p_{\boldsymbol{\epsilon}}$.
* A single-pass network is already a completely general pushforward. **You do not need an ODE to be expressive.**
  Diffusion's iterative sampler buys you a *tractable training signal*, not extra expressiveness.

## 2. Two ways to build the map

Both diffusion and drifting build $f$ as a *composition* of many small moves $\mathbf{x} \leftarrow \mathbf{x}+\Delta\mathbf{x}$.
The difference is **which index the composition runs over**.

| | diffusion / flow matching | drifting |
|---|---|---|
| the increment | $\Delta\mathbf{x}_i$ from a learned velocity/score at noise level $t_i$ | $\Delta\mathbf{x}_i = f_{i+1}(\boldsymbol{\epsilon})-f_i(\boldsymbol{\epsilon})$, caused by the parameter update |
| index $i$ runs over | **inference** steps | **training** steps |
| number of increments | 50–1000 network evaluations, every time you sample | tens of thousands of SGD steps, once, ever |
| inference cost | $N$ NFE | **1 NFE** |
| what the net outputs | a velocity at a noise level | the sample itself |

Diffusion decomposes a hard transport problem into a chain of easy ones **and pays for it at inference forever**.
Drifting says: *training is already an iterative process — use that iteration to do the transport.*

The consequence is that a drifting generator has **no time variable at all**. $f_\theta(\boldsymbol{\epsilon})$ is the sample.

In [ ]:
# The same total transport, split two different ways.
target = torch.tensor([[2.5, 1.0]])
start  = torch.tensor([[-2.5, -1.0]])
steps  = 8
path   = start + (target - start) * torch.linspace(0, 1, steps + 1)[:, None]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(path[:, 0], path[:, 1], "-o", c="tab:blue")
ax[0].set_title("diffusion: 8 increments at INFERENCE\n(8 network calls per sample)")
ax[1].plot(path[:, 0], path[:, 1], "-o", c="tab:orange")
for i in range(steps + 1):
    ax[1].annotate(f"$f_{{{i}}}$", path[i].tolist(), fontsize=7,
                   textcoords="offset points", xytext=(0, 7))
ax[1].set_title("drifting: 8 increments at TRAINING\n(1 network call per sample, forever)")
for a in ax:
    a.scatter(*start[0], marker="s", s=70, c="k"); a.scatter(*target[0], marker="*", s=200, c="crimson")
    a.set_aspect("equal"); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Measuring the gap between $q$ and $p$ from samples alone

We need a signal that tells us how far $q$ is from $p$ when we only have **samples** from each. The classical options:

**KL divergence** $\;\mathrm{KL}(q\|p)=\mathbb{E}_q[\log q - \log p]$ — needs both densities. Unavailable.

**Wasserstein-2** $\;W_2^2(q,p)=\inf_{\gamma}\mathbb{E}_{(x,y)\sim\gamma}\|x-y\|^2$ over couplings $\gamma$ — estimable
from samples but requires solving an optimal-transport problem per batch.

**Maximum Mean Discrepancy** with kernel $\xi$:
$$\mathrm{MMD}^2(q,p)=\mathbb{E}_{q,q}[\xi(\mathbf{x},\mathbf{x}')]-2\,\mathbb{E}_{q,p}[\xi(\mathbf{x},\mathbf{y})]+\mathbb{E}_{p,p}[\xi(\mathbf{y},\mathbf{y}')]$$
This one *is* a plain function of samples. Let's compute it and watch it detect a mismatch.

In [ ]:
def mmd2(X, Y, sigma=1.0):
    def k(A, B):
        return torch.exp(-torch.cdist(A, B) ** 2 / (2 * sigma ** 2))
    n, m = X.shape[0], Y.shape[0]
    kxx = (k(X, X).sum() - n) / (n * (n - 1))
    kyy = (k(Y, Y).sum() - m) / (m * (m - 1))
    kxy = k(X, Y).mean()
    return kxx + kyy - 2 * kxy

def bimodal(n, sep=2.0, sd=0.45):
    idx = torch.randint(0, 2, (n,))
    c = torch.tensor([[-sep, 0.0], [sep, 0.0]])
    return c[idx] + sd * torch.randn(n, 2)

P = bimodal(512)
seps = torch.linspace(0.0, 4.0, 25)
vals = [mmd2(bimodal(512, sep=s.item()), P).item() for s in seps]
plt.figure(figsize=(6, 3.2))
plt.plot(seps, vals, "-o", ms=3)
plt.axvline(2.0, ls="--", c="k", label="true separation")
plt.xlabel("separation of q's modes"); plt.ylabel("MMD$^2$(q, p)")
plt.title("MMD is a sample-only discrepancy, minimised at q = p")
plt.legend(); plt.grid(alpha=0.3); plt.show()

MMD works, and moment-matching generative models are built on exactly this. But note what happened: we produced a
**scalar**. To train with it we differentiate the scalar and hope the gradient is informative.

The drifting paper takes the other route. It never defines a discrepancy. It directly writes down the **vector field
that should move each sample**, and asks only that the field vanish at the solution. The connection is
(`THEORY.md` §8; paper Appendix C.2): any discrepancy $\mathcal{L}$ induces a field
$\mathbf{V}=-\tfrac12\partial\mathcal{L}/\partial\mathbf{x}$, but not every useful field comes from a discrepancy —
and the normalised (mean-shift) field, which works far better in practice, is one that doesn't.

Designing the *field* rather than the *scalar* is the paper's central methodological move.

## 4. The continuity equation and Wasserstein gradient flow

If particles move with velocity $v_t$, their density obeys the **continuity equation**
$$\partial_t q_t + \nabla\!\cdot\!(q_t v_t)=0$$
This is mass conservation, identical to the fluid-dynamics equation. Given a functional $\mathcal{F}(q)$, the
**Wasserstein-2 gradient flow** is the steepest-descent curve in the space of distributions; its velocity is
$$v_t(\mathbf{x}) = -\nabla_{\mathbf{x}}\frac{\delta\mathcal{F}}{\delta q_t}(\mathbf{x})$$
For $\mathcal{F}(q)=\mathrm{KL}(q\|p)$ the first variation is $\log(q/p)+1$, so
$$\boxed{\,v_t(\mathbf{x}) = \nabla\log p(\mathbf{x}) - \nabla\log q_t(\mathbf{x})\,}$$

**Attraction to $p$ minus repulsion from $q$.** That is the drifting field, before any kernel appears. Substituting
KDE estimates for the two unavailable scores, and using the Gaussian score identity from Notebook 0a
($\tau^2\nabla\log\rho_\tau = $ mean shift), gives

$$\tau^{2}\,v_t(\mathbf{x}) \;\approx\; \mathbf{V}^{+}_{p}(\mathbf{x}) - \mathbf{V}^{-}_{q}(\mathbf{x})$$

So with a Gaussian kernel, **drifting is a KDE-approximated Wasserstein gradient flow of $\mathrm{KL}(q\|p)$**,
and $\mathrm{KL}$ decreases monotonically along it. Let's watch that happen.

In [ ]:
def kde_score(z, samples, tau):
    d2 = ((z[:, None, :] - samples[None, :, :]) ** 2).sum(-1)
    w = (-d2 / (2 * tau ** 2)).softmax(-1)
    return (w @ samples - z) / tau ** 2          # = grad log rho_tau

def wgf(q0, p, steps=200, eta=0.08, tau=0.6):
    q = q0.clone(); hist = [q.clone()]
    for _ in range(steps):
        # leave-one-out for the self-score so a particle is not repelled by itself
        d2 = ((q[:, None, :] - q[None, :, :]) ** 2).sum(-1) + torch.eye(q.shape[0]) * 1e9
        w = (-d2 / (2 * tau ** 2)).softmax(-1)
        s_q = (w @ q - q) / tau ** 2
        v = kde_score(q, p, tau) - s_q
        q = q + eta * tau ** 2 * v
        hist.append(q.clone())
    return torch.stack(hist)

H = wgf(0.3 * torch.randn(400, 2), P)
kl_proxy = [mmd2(h, P).item() for h in H]

fig = plt.figure(figsize=(14, 3.2))
for i, t in enumerate([0, 10, 40, 100, 200]):
    a = fig.add_subplot(1, 6, i + 1)
    a.scatter(P[:, 0], P[:, 1], s=3, c="tab:blue", alpha=0.3)
    a.scatter(H[t][:, 0], H[t][:, 1], s=3, c="tab:orange")
    a.set_title(f"t={t}", fontsize=9); a.set_xlim(-4, 4); a.set_ylim(-2.5, 2.5)
    a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([])
a = fig.add_subplot(1, 6, 6)
a.semilogy(kl_proxy); a.set_title("MMD$^2$(q$_t$, p)", fontsize=9); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Monotone decrease, no mode collapse. This is the theoretical backbone: in the population/Gaussian limit drifting is
a gradient flow with a Lyapunov function. The Laplace kernel the paper actually uses breaks the exact gradient-flow
structure (Notebook 0a §6), but the intuition survives, and the mechanism —
**attract to data, repel from self, stop when they cancel** — is the same.

## 5. Sampling error: why batch size matters so much

Every field we compute is a *finite-sample estimate*. With $N_{\text{pos}}$ positives and $N_{\text{neg}}$ negatives, the
estimated field has variance $\sim 1/N$. This is why Table 2 of the paper shows FID improving monotonically as
$N_{\text{pos}}$ and $N_{\text{neg}}$ grow at **fixed** total compute — and it is the same phenomenon as "more negatives
help" in contrastive learning.

In [ ]:
ref = bimodal(20000)
probe = torch.tensor([[0.0, 0.0], [1.5, 0.3], [-2.0, 0.5]])

def ms(x, S, tau=0.7):
    return (-torch.cdist(x, S) / tau).softmax(-1) @ S - x

V_true = ms(probe, ref)
Ns = [4, 8, 16, 32, 64, 128, 256, 512, 1024]
err = []
for n in Ns:
    e = torch.stack([(ms(probe, bimodal(n)) - V_true).norm(dim=-1) for _ in range(200)])
    err.append(e.mean().item())

plt.figure(figsize=(5.5, 3.2))
plt.loglog(Ns, err, "-o", label="empirical error")
plt.loglog(Ns, err[0] * (torch.tensor(Ns).float() / Ns[0]) ** -0.5, "--", label=r"$N^{-1/2}$")
plt.xlabel("number of positive samples"); plt.ylabel("|V_hat - V|")
plt.title("the drift estimate is a Monte-Carlo estimate"); plt.legend(); plt.grid(alpha=0.3, which="both")
plt.show()

Clean $N^{-1/2}$. Practical consequence: **drifting wants big batches.** The paper's final runs use an effective
batch of 8192 built as $N_c{=}128$ class labels $\times$ $N_{\text{neg}}{=}64$ generated samples, with 128 positives
per class. Notebook 3 covers how to structure that.

## Summary

* A generator is a pushforward map $q=f_{\#}p_{\boldsymbol{\epsilon}}$; sampling costs one call.
* Diffusion iterates at inference, drifting iterates at training. Same total transport, different bill.
* You can only access $p$ and $q$ through samples. Drifting designs a **vector field** rather than a scalar discrepancy.
* In the Gaussian/population limit that field is the Wasserstein gradient flow velocity of $\mathrm{KL}(q\|p)$:
  $\nabla\log p - \nabla\log q$.
* Field estimates converge at $N^{-1/2}$, so batch composition is a first-class design choice.

Next: [0c. Fixed Points, Stop-Gradients, and Self-Distillation](<0c. Fixed Points, Stop-Gradients, and Self-Distillation.ipynb>).